In [19]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import altair as alt


In [20]:
old_df = pd.read_parquet("../data/processed/merged.parquet")

In [21]:
df = pd.read_parquet("../data/processed/processed.parquet")

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17351 entries, 0 to 17350
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   parent_asin             17351 non-null  str    
 1   product_title           4764 non-null   str    
 2   features                4764 non-null   object 
 3   description             4764 non-null   object 
 4   categories              4764 non-null   object 
 5   details                 4764 non-null   object 
 6   price                   3246 non-null   float64
 7   derived_avg_rating      17351 non-null  float64
 8   max_helpful_vote        17351 non-null  int64  
 9   n_reviews               17351 non-null  int64  
 10  review_docs             17351 non-null  object 
 11  review_text             17351 non-null  str    
 12  candidate_review_title  17351 non-null  str    
 13  candidate_review_text   17351 non-null  str    
dtypes: float64(2), int64(2), object(5), str(5)
memory

In [23]:
df.head()

,parent_asin,product_title,features,description,categories,details,price,derived_avg_rating,max_helpful_vote,n_reviews,review_docs,review_text,candidate_review_title,candidate_review_text
0,B0BX9GK3PB,"Midea MRU05M2AWW Upright Freezer, 5.3 Cu.ft, w...",[Reversible Door Hinge - allows right or left ...,[Midea¡¯s 5.3 Cubic Feet Upright Freezer is th...,"[Appliances, Refrigerators, Freezers & Ice Mak...","[(Manufacturer, ""Midea""), (Part Number, ""1""), ...",369.99,4.266667,12,45,[Great product Good value: Fit my space and ne...,Great product Good value: Fit my space and nee...,Damaged,Came damaged. The seal was terrible. The whole...
1,B0B9NFXVST,Fasezoomit Compatible Charcoal Water Filter Re...,[【FITS BREVILLE MACHINES 】Universal compatible...,[],"[Small Appliance Parts & Accessories, Coffee &...","[(Package Dimensions, ""6.1 x 5.12 x 2.28 inche...",9.75,4.000000,0,4,[Amazing how coffee tastes when you use great ...,Amazing how coffee tastes when you use great F...,Amazing how coffee tastes when you use great F...,Why do folks pay so much for Coffee House coff...
2,B08PTXKTPZ,"GinsonWare SET OF 4, ROUND STOVE TOP BURNER CO...",[Comes with 2pcs 8 inch and 2 pcs 10 inch roun...,[],"[Appliances, Parts & Accessories, Range Parts ...","[(Package Dimensions, ""11.02 x 10.16 x 0.94 in...",17.99,3.000000,2,3,[love the look: This burner cover is great to ...,love the look: This burner cover is great to k...,love the look,This burner cover is great to keep food from f...
3,B0BTSQRXNZ,Aprilaire 35 Water Panel Humidifier Filter Rep...,[BUY WITH CONFIDENCE This genuine replacement ...,[Maintaining proper humidity levels is integra...,"[Appliances, Parts & Accessories, Humidifier P...","[(Brand, ""Aprilaire""), (Special Feature, ""Cera...",26.49,4.808333,1,120,[Good filter for my inhome system: It fits per...,Good filter for my inhome system: It fits perf...,New filter every year!,until recently I did not know April Air needed...
4,B091V1ZYJM,Stove Countertop Gap Covers - Heavy Duty Stain...,[【APPLY TO DIFFERENT SITUATIONS】 - This stove ...,[Stove Countertop Gap Covers - Heat Resistant ...,"[Appliances, Parts & Accessories, Range Parts ...","[(Product Dimensions, ""25.3 x 1 x 3.5 inches"")...",39.99,5.000000,4,3,[Totally helped with my problem: The gap was 1...,Totally helped with my problem: The gap was 1”...,Finally fixed the gap issue and replaced the s...,"When our range died and we replaced it, the ne..."


In [24]:
missingness = pd.DataFrame(
    {"n_missing": df.isna().sum(), "prop_missing": df.isna().mean()}
).sort_values("prop_missing", ascending=False)

missingness

,n_missing,prop_missing
price,14105,0.812921
product_title,12587,0.725434
features,12587,0.725434
description,12587,0.725434
categories,12587,0.725434
details,12587,0.725434
parent_asin,0,0.000000
derived_avg_rating,0,0.000000
max_helpful_vote,0,0.000000
n_reviews,0,0.000000


In [25]:
df.describe()

,price,derived_avg_rating,max_helpful_vote,n_reviews
count,3246.000000,17351.000000,17351.000000,17351.000000
mean,71.172723,4.284624,2.866694,3.458014
std,238.458754,1.147784,17.412706,11.699019
min,1.940000,1.000000,0.000000,1.000000
25%,13.990000,4.000000,0.000000,1.000000
50%,23.965000,5.000000,0.000000,1.000000
75%,44.997500,5.000000,1.000000,3.000000
max,6999.000000,5.000000,1138.000000,685.000000
